# 03 · EV Fleet Forecasting — Washington State

**Dataset**: WA State DOL Electric Vehicle Population Data — 177,866 records, ~99.8% WA registrations.

This notebook builds two forecasting approaches:

- **ARIMA(2,1,1)** — classical statistical baseline on annual fleet counts
- **LightGBM** — lag-feature ML model for iterative multi-step forecasting

> **Known limitation**: both models operate on *model-year fleet stock* (vehicles currently
> registered by model year), not calendar-year adoption flow. Forecasts should be read as
> projected fleet size by model year, not as new annual registrations.

In [2]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

from src.features import full_preprocessing_pipeline, YEAR_COL, MAKE_COL, EV_TYPE_COL, RANGE_COL

DATA_PATH = Path("../data/raw/Electric_Vehicle_Population_Data.csv")
df = full_preprocessing_pipeline(DATA_PATH)

# Focus on model years with reliable data (2010–2023)
df_fleet = df[(df[YEAR_COL] >= 2010) & (df[YEAR_COL] <= 2023)].copy()
print(f"Fleet records (2010–2023): {len(df_fleet):,}")

Fleet records (2010–2023): 170,749


## 1 · Annual time series

In [3]:
# Total fleet by model year
ts = (
    df_fleet.groupby(YEAR_COL)
    .size()
    .reset_index(name="count")
    .sort_values(YEAR_COL)
)
ts["yoy_growth"] = ts["count"].pct_change() * 100
print(ts.to_string(index=False))
print(f"\nSeries length: {len(ts)} annual observations (2010–2023)")

 Model Year  count  yoy_growth
       2010     23         NaN
       2011    775 3269.565217
       2012   1618  108.774194
       2013   4409  172.496910
       2014   3509  -20.412792
       2015   4844   38.045027
       2016   5483   13.191577
       2017   8562   56.155389
       2018  14323   67.285681
       2019  10940  -23.619353
       2020  11768    7.568556
       2021  19132   62.576479
       2022  27776   45.180849
       2023  57587  107.326469

Series length: 14 annual observations (2010–2023)


## 2 · Fleet size and YoY growth

In [4]:
from plotly.subplots import make_subplots

fig_combined = make_subplots(specs=[[{"secondary_y": True}]])
fig_combined.add_bar(
    x=ts[YEAR_COL], y=ts["count"],
    name="Fleet Size", marker_color="#1f77b4", opacity=0.7,
)
fig_combined.add_scatter(
    x=ts[YEAR_COL], y=ts["yoy_growth"],
    mode="lines+markers", name="YoY Growth %",
    line=dict(color="crimson", width=2),
    secondary_y=True,
)
fig_combined.update_layout(
    title="EV Fleet Size and YoY Growth Rate by Model Year",
    xaxis_title="Model Year",
    template="plotly_white",
)
fig_combined.update_yaxes(title_text="Fleet Count", secondary_y=False)
fig_combined.update_yaxes(title_text="YoY Growth (%)", secondary_y=True)
fig_combined.show()

## 3 · ARIMA(2,1,1) baseline

Train on 2010–2021, evaluate on 2022–2023, forecast 2024–2025.

> ARIMA assumes a locally linear trend. EV adoption is exponential from ~2019 onward,
> so expect systematic underestimation in the test period. Treat this as a conservative
> lower-bound baseline only.

In [5]:
try:
    from statsmodels.tsa.arima.model import ARIMA
    HAS_STATSMODELS = True
except ImportError:
    print("statsmodels not installed. Run: pip install statsmodels")
    HAS_STATSMODELS = False

if HAS_STATSMODELS:
    ts_indexed = ts.set_index(YEAR_COL)["count"]

    # Convert integer year index to DatetimeIndex with annual frequency.
    # statsmodels requires DatetimeIndex or PeriodIndex — plain integers
    # produce ValueWarning and break get_forecast().
    ts_indexed.index = pd.DatetimeIndex(
        pd.to_datetime(ts_indexed.index, format="%Y"),
        freq="YS",  # Year-Start — unambiguous, fully supported by statsmodels
    )

    train = ts_indexed[ts_indexed.index.year <= 2021]
    test  = ts_indexed[ts_indexed.index.year > 2021]

    model_arima = ARIMA(train, order=(2, 1, 1)).fit()
    forecast    = model_arima.get_forecast(steps=len(test) + 2)  # +2 for 2024, 2025
    fc_mean     = forecast.predicted_mean
    fc_ci       = forecast.conf_int(alpha=0.05)

    # DatetimeIndex → integer years for Plotly axis labels
    years_forecast = [p.year for p in fc_mean.index]
    train_years    = [p.year for p in train.index]
    test_years     = [p.year for p in test.index]

    # Test-set accuracy
    from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
    test_preds = fc_mean[[p.year in test_years for p in fc_mean.index]]
    arima_mae  = mean_absolute_error(test.values, test_preds.values)
    arima_mape = mean_absolute_percentage_error(test.values, test_preds.values) * 100
    print(f"ARIMA test MAE : {arima_mae:,.0f} vehicles")
    print(f"ARIMA test MAPE: {arima_mape:.1f}%  (directional bias: underestimates exponential growth)")

    fig = go.Figure()
    fig.add_scatter(
        x=train_years, y=train.values,
        mode="lines+markers", name="Historical",
        line=dict(color="#1f77b4"),
    )
    fig.add_scatter(
        x=test_years, y=test.values,
        mode="lines+markers", name="Actual (test)",
        line=dict(color="green"),
    )
    fig.add_scatter(
        x=years_forecast, y=fc_mean.values,
        mode="lines+markers", name="ARIMA Forecast",
        line=dict(color="crimson", dash="dash"),
    )
    fig.add_traces(go.Scatter(
        x=years_forecast + years_forecast[::-1],
        y=fc_ci.iloc[:, 1].tolist() + fc_ci.iloc[:, 0].tolist()[::-1],
        fill="toself", fillcolor="rgba(220,50,50,0.15)",
        line=dict(width=0), name="95% CI",
    ))
    fig.update_layout(
        title="ARIMA(2,1,1) EV Fleet Forecast",
        xaxis_title="Model Year",
        yaxis_title="Fleet Size",
        template="plotly_white",
    )
    fig.show()

    print(f"\nForecast summary:")
    for yr, val in zip(years_forecast, fc_mean):
        print(f"  {int(yr)}: {val:,.0f}")

ARIMA test MAE : 25,773 vehicles
ARIMA test MAPE: 53.8%  (directional bias: underestimates exponential growth)

Forecast summary:
  2022: 18,019
  2023: 15,798
  2024: 17,669
  2025: 17,352


## 4 · LightGBM lag-feature forecast

Features: lags 1–3 and 3-period rolling mean of fleet count, plus model year as trend proxy.

> **Known limitation**: tree-based models cannot extrapolate beyond the training range.
> Iterative forecasts will converge to a leaf bound. Results are illustrative only;
> see ARIMA or a log-linear trend model for more reliable extrapolation.

In [6]:
try:
    import lightgbm as lgb
    from sklearn.metrics import mean_absolute_error
    HAS_LGB = True
except ImportError:
    print("lightgbm not installed. Run: pip install lightgbm")
    HAS_LGB = False

if HAS_LGB:
    # Build lag-feature DataFrame from the 14-row annual series
    ts_ml = ts.set_index(YEAR_COL)["count"].copy()
    df_ml = pd.DataFrame({"count": ts_ml})
    for lag in [1, 2, 3]:
        df_ml[f"lag_{lag}"] = df_ml["count"].shift(lag)
    df_ml["rolling_mean_3"] = df_ml["count"].shift(1).rolling(3).mean()
    df_ml["year"] = df_ml.index
    df_ml = df_ml.dropna()
    print(f"Usable rows after lag construction: {len(df_ml)}  (expect ~10)")

    FEATURES = ["lag_1", "lag_2", "lag_3", "rolling_mean_3", "year"]
    TARGET   = "count"

    split_year = 2020
    train_ml = df_ml[df_ml.index <= split_year]
    test_ml  = df_ml[df_ml.index > split_year]
    print(f"Train rows: {len(train_ml)}  |  Test rows: {len(test_ml)}")

    model_lgb = lgb.LGBMRegressor(
        n_estimators=200, learning_rate=0.05,
        num_leaves=8, random_state=42, verbose=-1,
    )
    model_lgb.fit(train_ml[FEATURES], train_ml[TARGET])

    pred_ml = model_lgb.predict(test_ml[FEATURES])
    mae_lgb = mean_absolute_error(test_ml[TARGET], pred_ml)
    print(f"LightGBM MAE on test set: {mae_lgb:.0f} vehicles")

    # Iterative forecast 2024–2026
    last_row = df_ml.iloc[-1].copy()
    forecasts = {}
    for yr in [2024, 2025, 2026]:
        feat = {
            "lag_1": last_row["count"],
            "lag_2": last_row["lag_1"],
            "lag_3": last_row["lag_2"],
            "rolling_mean_3": (
                last_row["count"] + last_row["lag_1"] + last_row["lag_2"]
            ) / 3,
            "year": yr,
        }
        pred = model_lgb.predict(pd.DataFrame([feat]))[0]
        forecasts[yr] = pred
        last_row = pd.Series(feat | {"count": pred})

    print("\nLightGBM iterative forecast (note: tree models cannot extrapolate):")
    for yr, val in forecasts.items():
        print(f"  {yr}: {val:,.0f}")

    fig_lgb = go.Figure()
    fig_lgb.add_scatter(
        x=list(df_ml.index), y=df_ml["count"].tolist(),
        mode="lines+markers", name="Historical",
        line=dict(color="#1f77b4"),
    )
    fig_lgb.add_scatter(
        x=list(forecasts.keys()), y=list(forecasts.values()),
        mode="lines+markers", name="LightGBM Forecast",
        line=dict(color="darkorange", dash="dash"),
        marker=dict(size=10),
    )
    fig_lgb.update_layout(
        title="LightGBM Lag-Feature EV Fleet Forecast (2024–2026)",
        xaxis_title="Model Year",
        yaxis_title="Fleet Size",
        template="plotly_white",
    )
    fig_lgb.show()

Usable rows after lag construction: 11  (expect ~10)
Train rows: 8  |  Test rows: 3
LightGBM MAE on test set: 26852 vehicles

LightGBM iterative forecast (note: tree models cannot extrapolate):
  2024: 7,980
  2025: 7,980
  2026: 7,980


## 5 · Adoption trajectory — top 5 makes

In [7]:
top5 = df_fleet[MAKE_COL].value_counts().head(5).index.tolist()
make_ts = (
    df_fleet[df_fleet[MAKE_COL].isin(top5)]
    .groupby([YEAR_COL, MAKE_COL])
    .size()
    .reset_index(name="count")
)

fig = px.line(
    make_ts, x=YEAR_COL, y="count", color=MAKE_COL,
    markers=True,
    title="Adoption Trajectory — Top 5 Makes (2010–2023)",
    labels={YEAR_COL: "Model Year", "count": "Fleet Count", MAKE_COL: "Make"},
    template="plotly_white",
)
fig.show()

## 6 · BEV penetration rate

In [8]:
bev_share = (
    df_fleet.groupby(YEAR_COL)["is_bev"]
    .mean()
    .reset_index()
)
bev_share["BEV Share %"] = bev_share["is_bev"] * 100

fig = px.area(
    bev_share, x=YEAR_COL, y="BEV Share %",
    title="BEV Penetration Rate by Model Year",
    labels={YEAR_COL: "Model Year"},
    template="plotly_white",
    color_discrete_sequence=["#1f77b4"],
)
fig.update_layout(yaxis_range=[0, 100])
fig.show()

print("\nBEV share by model year:")
print(bev_share.to_string(index=False))


BEV share by model year:
 Model Year   is_bev  BEV Share %
       2010 0.869565    86.956522
       2011 0.901935    90.193548
       2012 0.468480    46.847960
       2013 0.642323    64.232252
       2014 0.495583    49.558279
       2015 0.732659    73.265896
       2016 0.681014    68.101404
       2017 0.528031    52.803083
       2018 0.695245    69.524541
       2019 0.817002    81.700183
       2020 0.847893    84.789259
       2021 0.801014    80.101401
       2022 0.844110    84.411002
       2023 0.875250    87.524962
